In [1]:
import numpy as np
import pandas as pd
from spreg import GM_Error
import libpysal
from libpysal.weights import KNN
import matplotlib.pyplot as plt
import geopandas as gpd
import scipy as sp
import arviz as az

RANDOM_SEED = 5781

# Transportation

## Data

In [2]:
tran_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ONR_epa_climate.parquet'
tran_df = gpd.read_parquet(tran_par_file)
tran_wgt_file = '/work/hawkins_lab/vulcan/results/tran_weights.csv'
tran_wgt_df = pd.read_csv(tran_wgt_file)
# Create columns that assign decile numbers for each treatment variable
tran_df['treat_density'] = tran_df['d1a'] /1000
tran_df['treat_diversity'] = tran_df['d2b_e5mixa']
tran_df['treat_design'] = tran_df['d3a']
tran_df['treat_distance'] = tran_df['d4a'].replace(-99999,1500)  /1000
tran_df['treat_destination'] = tran_df['d5ar']  /1000

tran_df = pd.concat([tran_df,tran_wgt_df], axis=1)
tran_df = tran_df[(tran_df["value_weig"]>0) & (tran_df["totpop"]>0)]
tran_df["d1aa_cbsa"] = tran_df["totpop_cbsa"]/tran_df["ac_unpr_cbsa"] # true cbsa density

In [3]:
tran_df.groupby('cbsa').size()

cbsa
1.11          7
1.13         19
1.19         19
1.23         15
1.25         24
           ... 
49660.00    520
49700.00    111
49740.00    140
49780.00     75
49820.00      9
Length: 2013, dtype: int64

In [4]:
for c in ["treat_density","treat_diversity","treat_design","treat_distance","treat_destination", 
          "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa","d3a_cbsa", "d4a_cbsa", "d5ar_cbsa"]:
    mu, sd = tran_df[c].mean(), tran_df[c].std()
    tran_df[c+"_z"] = (tran_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div","des","dist","dest"]:
    mu, sd = tran_df[g].mean(), tran_df[g].std()
    tran_df["gps_"+g+"_z"] = (tran_df[g] - mu) / (sd if sd!=0 else 1.0)

tran_df["treat_density_ps"]   = tran_df["treat_density_z"]   * tran_df["gps_dens_z"]
tran_df["treat_diversity_ps"]  = tran_df["treat_diversity_z"] * tran_df["gps_div_z"]
tran_df["treat_design_ps"]     = tran_df["treat_design_z"]    * tran_df["gps_des_z"]
tran_df["treat_distance_ps"]  = tran_df["treat_distance_z"]  * tran_df["gps_dist_z"]
tran_df["treat_destination_ps"]= tran_df["treat_destination_z"] * tran_df["gps_dest_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              "d3a_cbsa_z",
              "d4a_cbsa_z", 
              "d5ar_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "gps_des_z",
              "gps_dist_z",
              "gps_dest_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_design_z",
              "treat_distance_z",
              "treat_destination_z",
              "treat_density_ps",
              "treat_diversity_ps",
              "treat_design_ps",
              "treat_distance_ps",
              "treat_destination_ps"]

X = tran_df.copy()

y = np.log(tran_df["value_weig"].replace(0, 10**-6) / tran_df["totpop"].replace(0, np.nan))

# X.loc[:,log_predictors] = np.log(X.loc[:,log_predictors].replace(0, 10**-6) ) # replace 0 with 10**-6

X1 = X.loc[:, predictors]
X_fe = pd.get_dummies(tran_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)

## Model KNN

In [5]:
W = KNN.from_dataframe(tran_df, k=12)
W.transform = 'r'

model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

In [6]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      215309
Mean dependent var  :      0.9804                Number of Variables   :         356
S.D. dependent var  :      1.1514                Degrees of Freedom    :      214953
Pseudo R-squared    :      0.2569

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.30316         0.06949        18.75262         0.00000
       totpop_cbsa_z        -0.02424         0.06196        -0.39119         0.69565
         d1aa_cbsa_z         0.25985         0.38049         0.68293

## Model diagnostics

In [7]:
from esda.moran import Moran

residuals = model.y - model.predy

mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.26443159005537625 0.001


## Model Distance Band

In [8]:
from libpysal.weights import DistanceBand

W = DistanceBand.from_dataframe(
    tran_df,
    threshold=10000,
    binary=False,
    alpha=-1.0   # exponential decay
)
W.transform = 'r'


model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/scipy/sparse/_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 7581 disconnected components.
 There are 5553 islands with ids: 27, 29, 30, 31, 32, 33, 34, 40, 109, 126, 129, 130, 131, 132, 133, 134, 156, 157, 160, 199, 202, 203, 204, 205, 211, 224, 317, 328, 390, 391, 392, 400, 401, 408, 409, 411, 412, 413, 415, 421, 422, 425, 426, 429, 436, 437, 449, 450, 465, 476, 539, 540, 541, 543, 546, 550, 556, 559, 562, 569, 605, 607, 608, 609, 610, 743, 745, 854, 855, 867, 869, 983, 1035, 1036, 1037, 1045, 1046, 1056, 1057, 1058, 1072, 1155, 1158, 1160, 1762, 1946, 1949, 1953, 1954, 1958, 1975, 1976, 1977, 2176, 2177, 2180, 2181, 2182, 2464, 2555, 2556, 2557, 2559, 2575, 2756, 2757, 2850, 

('WARNING: ', 27, ' is an island (no neighbors)')
('WARNING: ', 29, ' is an island (no neighbors)')
('WARNING: ', 30, ' is an island (no neighbors)')
('WARNING: ', 31, ' is an island (no neighbors)')
('WARNING: ', 32, ' is an island (no neighbors)')
('WARNING: ', 33, ' is an island (no neighbors)')
('WARNING: ', 34, ' is an island (no neighbors)')
('WARNING: ', 40, ' is an island (no neighbors)')
('WARNING: ', 109, ' is an island (no neighbors)')
('WARNING: ', 126, ' is an island (no neighbors)')
('WARNING: ', 129, ' is an island (no neighbors)')
('WARNING: ', 130, ' is an island (no neighbors)')
('WARNING: ', 131, ' is an island (no neighbors)')
('WARNING: ', 132, ' is an island (no neighbors)')
('WARNING: ', 133, ' is an island (no neighbors)')
('WARNING: ', 134, ' is an island (no neighbors)')
('WARNING: ', 156, ' is an island (no neighbors)')
('WARNING: ', 157, ' is an island (no neighbors)')
('WARNING: ', 160, ' is an island (no neighbors)')
('WARNING: ', 199, ' is an island (no n

In [9]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      215309
Mean dependent var  :      0.9804                Number of Variables   :         356
S.D. dependent var  :      1.1514                Degrees of Freedom    :      214953
Pseudo R-squared    :      0.2603

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.51683         0.05502        27.57019         0.00000
       totpop_cbsa_z        -0.04538         0.05672        -0.80006         0.42367
         d1aa_cbsa_z         0.17220         0.33888         0.50813

## Model diagnostics

In [10]:
from esda.moran import Moran

residuals = model.y - model.predy

mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.1698148023011235 0.001


# Residential Energy

## Data

In [11]:
res_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_RES_epa_climate.parquet'
res_df = gpd.read_parquet(res_par_file)
res_wgt_file = '/work/hawkins_lab/vulcan/results/res_weights.csv'
res_wgt_df = pd.read_csv(res_wgt_file)
# Create columns that assign decile numbers for each treatment variable
res_df['treat_density'] = res_df['d1a']
res_df['treat_diversity'] = res_df['d2b_e5mixa']

res_df = pd.concat([res_df,res_wgt_df], axis=1)
res_df = res_df[(res_df["value_weig"]>0) & (res_df["totpop"]>0)]
res_df["d1aa_cbsa"] = res_df["totpop_cbsa"]/res_df["ac_unpr_cbsa"] # true cbsa density

for c in ["treat_density","treat_diversity", "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa"]:
    mu, sd = res_df[c].mean(), res_df[c].std()
    res_df[c+"_z"] = (res_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div"]:
    mu, sd = res_df[g].mean(), res_df[g].std()
    res_df["gps_"+g+"_z"] = (res_df[g] - mu) / (sd if sd!=0 else 1.0)

res_df["treat_density_ps"]   = res_df["treat_density_z"]   * res_df["gps_dens_z"]
res_df["treat_diversity_ps"]  = res_df["treat_diversity_z"] * res_df["gps_div_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_density_ps",
              "treat_diversity_ps"]

X = res_df.copy()

y = np.log(res_df["value_weig"].replace(0, 10**-6) / res_df["totpop"].replace(0, np.nan)) # make per capita

X1 = X.loc[:, predictors]
X_fe = pd.get_dummies(res_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)

## Model KNN

In [12]:
W = KNN.from_dataframe(res_df, k=12)
W.transform = 'r'

model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

In [13]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      215249
Mean dependent var  :     -0.1919                Number of Variables   :         344
S.D. dependent var  :      1.2203                Degrees of Freedom    :      214905
Pseudo R-squared    :      0.6353

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        -1.10818         0.04165       -26.60883         0.00000
       totpop_cbsa_z         0.01535         0.02122         0.72342         0.46942
         d1aa_cbsa_z         0.14493         0.18120         0.79985

## Model diagnostics

In [14]:
from esda.moran import Moran

residuals = model.y - model.predy

mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.22577471223091233 0.001


## Model Distance Band

In [15]:
from libpysal.weights import DistanceBand

W = DistanceBand.from_dataframe(
    res_df,
    threshold=10000,
    binary=False,
    alpha=-1.0   # exponential decay
)
W.transform = 'r'


model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/scipy/sparse/_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 7577 disconnected components.
 There are 5551 islands with ids: 27, 29, 30, 31, 32, 33, 34, 40, 109, 126, 129, 130, 131, 132, 133, 134, 156, 157, 160, 199, 202, 203, 204, 205, 211, 224, 317, 328, 390, 391, 392, 400, 401, 408, 409, 411, 412, 413, 415, 421, 422, 425, 426, 429, 436, 437, 449, 450, 465, 476, 539, 540, 541, 543, 546, 550, 556, 559, 562, 569, 605, 607, 608, 609, 610, 743, 745, 854, 855, 867, 869, 983, 1035, 1036, 1037, 1045, 1046, 1056, 1057, 1058, 1072, 1155, 1158, 1160, 1762, 1946, 1949, 1953, 1954, 1958, 1975, 1976, 1977, 2176, 2177, 2180, 2181, 2182, 2464, 2555, 2556, 2557, 2559, 2575, 2756, 2757, 2850, 

('WARNING: ', 27, ' is an island (no neighbors)')
('WARNING: ', 29, ' is an island (no neighbors)')
('WARNING: ', 30, ' is an island (no neighbors)')
('WARNING: ', 31, ' is an island (no neighbors)')
('WARNING: ', 32, ' is an island (no neighbors)')
('WARNING: ', 33, ' is an island (no neighbors)')
('WARNING: ', 34, ' is an island (no neighbors)')
('WARNING: ', 40, ' is an island (no neighbors)')
('WARNING: ', 109, ' is an island (no neighbors)')
('WARNING: ', 126, ' is an island (no neighbors)')
('WARNING: ', 129, ' is an island (no neighbors)')
('WARNING: ', 130, ' is an island (no neighbors)')
('WARNING: ', 131, ' is an island (no neighbors)')
('WARNING: ', 132, ' is an island (no neighbors)')
('WARNING: ', 133, ' is an island (no neighbors)')
('WARNING: ', 134, ' is an island (no neighbors)')
('WARNING: ', 156, ' is an island (no neighbors)')
('WARNING: ', 157, ' is an island (no neighbors)')
('WARNING: ', 160, ' is an island (no neighbors)')
('WARNING: ', 199, ' is an island (no n

In [16]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      215249
Mean dependent var  :     -0.1919                Number of Variables   :         344
S.D. dependent var  :      1.2203                Degrees of Freedom    :      214905
Pseudo R-squared    :      0.6363

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        -1.11409         0.03768       -29.56975         0.00000
       totpop_cbsa_z         0.02714         0.02049         1.32471         0.18527
         d1aa_cbsa_z         0.01905         0.17254         0.11039

## Model diagnostics

In [17]:
from esda.moran import Moran

residuals = model.y - model.predy

mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.17016789114015796 0.001


## Residential Electricity

## Data

In [18]:
elec_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_SC2_epa_climate.parquet'
elec_df = gpd.read_parquet(elec_par_file)
elec_wgt_file = '/work/hawkins_lab/vulcan/results/elec_weights.csv'
elec_wgt_df = pd.read_csv(elec_wgt_file)
# Create columns that assign decile numbers for each treatment variable
elec_df['treat_density'] = elec_df['d1a']
elec_df['treat_diversity'] = elec_df['d2b_e5mixa']

elec_df = pd.concat([elec_df,elec_wgt_df], axis=1)
elec_df = elec_df[(elec_df["res_tc"]>0) & (elec_df["totpop"]>0)]
elec_df["d1aa_cbsa"] = elec_df["totpop_cbsa"]/elec_df["ac_unpr_cbsa"] # true cbsa density

for c in ["treat_density","treat_diversity", "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa"]:
    mu, sd = elec_df[c].mean(), elec_df[c].std()
    elec_df[c+"_z"] = (elec_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div"]:
    mu, sd = elec_df[g].mean(), elec_df[g].std()
    elec_df["gps_"+g+"_z"] = (elec_df[g] - mu) / (sd if sd!=0 else 1.0)

elec_df["treat_density_ps"]   = elec_df["treat_density_z"]   * elec_df["gps_dens_z"]
elec_df["treat_diversity_ps"]  = elec_df["treat_diversity_z"] * elec_df["gps_div_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_density_ps",
              "treat_diversity_ps"]

X = elec_df.copy()

y = np.log(elec_df["res_tc"].replace(0, 10**-6) / elec_df["totpop"].replace(0, np.nan)) # make per capita

X1 = X.loc[:, predictors]
X_fe = pd.get_dummies(elec_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)

## Model KNN

In [19]:
W = KNN.from_dataframe(elec_df, k=12)
W.transform = 'r'

model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


In [20]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      216623
Mean dependent var  :     -0.9630                Number of Variables   :         349
S.D. dependent var  :      0.8472                Degrees of Freedom    :      216274
Pseudo R-squared    :      0.6388

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        -0.15325         0.03395        -4.51402         0.00001
       totpop_cbsa_z        -0.03154         0.01510        -2.08926         0.03668
         d1aa_cbsa_z         0.18125         0.12957         1.39893

## Model diagnostics

In [21]:
from esda.moran import Moran

residuals = model.y - model.predy

mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.40738102403229715 0.001


## Model Distance Band

In [22]:
from libpysal.weights import DistanceBand

W = DistanceBand.from_dataframe(
    elec_df,
    threshold=10000,
    binary=False,
    alpha=-1.0   # exponential decay
)
W.transform = 'r'


model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/scipy/sparse/_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 7839 disconnected components.
 There are 5830 islands with ids: 27, 29, 31, 32, 33, 34, 40, 42, 109, 125, 128, 129, 130, 131, 132, 133, 155, 159, 198, 199, 200, 201, 202, 203, 204, 210, 223, 316, 327, 379, 389, 390, 391, 399, 400, 407, 408, 409, 410, 411, 412, 414, 420, 421, 423, 424, 425, 428, 435, 436, 447, 448, 449, 464, 475, 538, 539, 540, 542, 544, 545, 549, 555, 558, 561, 568, 604, 606, 607, 608, 609, 742, 744, 853, 854, 866, 868, 982, 1034, 1035, 1036, 1044, 1045, 1055, 1056, 1057, 1071, 1154, 1157, 1159, 1761, 1821, 1824, 1945, 1948, 1952, 1953, 1957, 1974, 1975, 1976, 2175, 2176, 2179, 2180, 2181, 2463, 2553, 

('WARNING: ', 27, ' is an island (no neighbors)')
('WARNING: ', 29, ' is an island (no neighbors)')
('WARNING: ', 31, ' is an island (no neighbors)')
('WARNING: ', 32, ' is an island (no neighbors)')
('WARNING: ', 33, ' is an island (no neighbors)')
('WARNING: ', 34, ' is an island (no neighbors)')
('WARNING: ', 40, ' is an island (no neighbors)')
('WARNING: ', 42, ' is an island (no neighbors)')
('WARNING: ', 109, ' is an island (no neighbors)')
('WARNING: ', 125, ' is an island (no neighbors)')
('WARNING: ', 128, ' is an island (no neighbors)')
('WARNING: ', 129, ' is an island (no neighbors)')
('WARNING: ', 130, ' is an island (no neighbors)')
('WARNING: ', 131, ' is an island (no neighbors)')
('WARNING: ', 132, ' is an island (no neighbors)')
('WARNING: ', 133, ' is an island (no neighbors)')
('WARNING: ', 155, ' is an island (no neighbors)')
('WARNING: ', 159, ' is an island (no neighbors)')
('WARNING: ', 198, ' is an island (no neighbors)')
('WARNING: ', 199, ' is an island (no n

In [23]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      216623
Mean dependent var  :     -0.9630                Number of Variables   :         349
S.D. dependent var  :      0.8472                Degrees of Freedom    :      216274
Pseudo R-squared    :      0.6305

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        -0.14396         0.02984        -4.82481         0.00000
       totpop_cbsa_z        -0.03026         0.01486        -2.03656         0.04169
         d1aa_cbsa_z         0.09965         0.12407         0.80314

## Model diagnostics

In [24]:
from esda.moran import Moran

residuals = model.y - model.predy

mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.36321900579416355 0.001
